# Zero-shot screening — evaluation against the diagnostic set

Instead of fine-tuning on labelled examples, the screening criterion is written out in
natural language and handed to an instruction-following model. Nothing is trained.

**Why this is worth testing here.** The fine-tuned DeBERTa reached macro F1 0.93 on
synthetic validation but 0.245 recall on real abstracts: it learned the shape of the
generated text, not the construct. A zero-shot model has never seen those synthetic
examples, so it cannot make that mistake. It also makes the criterion auditable — the
prompt *is* the protocol, and a reviewer can read it and disagree with it.

**Precedent.** Costa et al. (2026) used a zero-shot NLI classifier for the smart-city
domain classification stage (Sect. 4.4), justifying it as scalable and as reducing the
need for domain-specific training. This notebook applies the same idea one stage
earlier, to binary screening.

**What is measured.** The same diagnostic set used for the fine-tuned model
(`diagnostic_test.json`, real corpus abstracts), so the two are directly comparable on
recall, precision and WSS@95.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================
import os, json, re, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

DATA_DIR = "../model/dataset"
OUT_DIR  = "../model/output"
os.makedirs(OUT_DIR, exist_ok=True)

POS = "Sensing of walkability dimension"
NEG = "Not related"

diag = pd.DataFrame(json.load(open(f"{DATA_DIR}/diagnostic_test.json", encoding="utf-8")))
diag["y"] = (diag["label"] == POS).astype(int)
print(f"Diagnostic set: {len(diag)} abstracts | {diag.y.mean()*100:.1f}% positive")

## 1. The screening protocol as a prompt

This is the artefact that replaces the training data. Everything the classifier knows
about the task is here, in text. The three boundary rules are the ones adjudicated on
real borderline cases; the worked examples are the hard negatives that a bag-of-words
model gets wrong.

Edit this cell — not a dataset — when the criterion changes.

In [ ]:
SYSTEM_PROMPT = """You screen abstracts for a systematic mapping review on how sensors \
and sensing technologies are used to measure dimensions of walkability in urban \
pedestrian environments.

## Decision
Return exactly one label:
- "Sensing of walkability dimension": the study uses a sensor or capture technology,
  acquired AT STREET LEVEL, to MEASURE an attribute of the pedestrian environment.
- "Not related": anything else.

## Conceptual frame
Walkability is the potential of the built environment to affect the propensity to walk
(Annunziata and Garau, 2020). Measurable dimensions follow Alfonzo's Hierarchy of
Walking Needs: accessibility (network, continuity, connectivity), safety (crossings,
traffic exposure, lighting, perceived safety), comfort (thermal, acoustic, air quality,
shade, surface), pleasurability (greenery, facades, enclosure, maintenance), and
feasibility (slope, steps, ramps, clear width).

## Boundary rules — decided on real cases, apply them literally

RULE 1 - STREET LEVEL DEFINES SCOPE.
The source that measures the PEDESTRIAN attribute determines eligibility. Purely aerial,
airborne or satellite acquisition is out of scope, even when it measures streetscape
features. Auxiliary aerial data is acceptable if the pedestrian attribute itself is
measured from street level.
  Airborne LiDAR voxel analysis of street furniture -> Not related.
  Aerial imagery for the road network PLUS street-level audit of sidewalk width and
  ramps -> Sensing.

RULE 2 - INSTRUMENT, NOT STIMULUS.
Imagery processed as measurement data is in scope. Imagery shown to respondents as
survey material is not, because nothing is being measured by an instrument.
  Stated-preference experiment using street-level images -> Not related.
  Segmentation of street imagery to compute green view index -> Sensing.

RULE 3 - THE INFERENTIAL TARGET MATTERS.
A street audit whose construct is not the pedestrian environment is out of scope, even
when the instrument is a standard walkability audit tool.
  Google Street View audit of 65 block characteristics to study opioid overdose risk
  -> Not related.
  Street view segmentation to predict property prices, carbon emissions, or crime
  -> Not related.

RULE 4 - SENSOR ON THE PERSON.
If the sensor measures the PERSON (steps, MVPA, physiology) and walkability comes from
an external index or GIS layer, it is Not related — the environment was not measured.
If the sensor measures the person in order to CHARACTERISE THE ENVIRONMENT (biosignals
mapped to street segments, gait data used to locate built-environment barriers), it is
Sensing.
  Accelerometer step counts plus a published walkability index -> Not related.
  Wearable electrodermal sensors mapped to segment-level streetscape attributes
  -> Sensing.

## Also Not related
- Walkability computed only from GIS or network analysis, with no sensor.
- Simulation instead of measurement (CFD, agent-based, microsimulation).
- Cyclists, e-scooters, drivers, robots or autonomous vehicles as the subject.
- Clinical gait, rehabilitation, laboratory instrumented walkways.
- Reviews that discuss sensing without collecting data.
- Papers that merely mention sensors while measuring by hand.

## Output
Return ONLY a JSON object, no markdown, no preamble:
{"label": "<one of the two labels>", "confidence": <0.0-1.0>, "reason": "<max 20 words>"}

`confidence` is your probability that the label is "Sensing of walkability dimension".
Use the full range: values near 0.5 mean genuinely borderline."""

print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

## 2. Calling the model

Set `ANTHROPIC_API_KEY` in your environment. The loop is deliberately simple: one
abstract per call, retries on failure, results cached to disk so an interrupted run
resumes instead of paying twice.

Cost scale: ~1,600 abstracts at roughly 1,000 input tokens each is a few US dollars.

In [ ]:
# pip install anthropic
from anthropic import Anthropic

client = Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"
CACHE  = f"{OUT_DIR}/zeroshot_cache.json"

cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def classify(text, retries=3):
    key = str(hash(text))
    if key in cache:
        return cache[key]
    for attempt in range(retries):
        try:
            r = client.messages.create(
                model=MODEL, max_tokens=200, temperature=0,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": f"ABSTRACT:\n{text[:6000]}"}],
            )
            raw = "".join(b.text for b in r.content if b.type == "text")
            raw = re.sub(r"```json|```", "", raw).strip()
            out = json.loads(raw)
            out["label"] = out["label"] if out.get("label") in (POS, NEG) else NEG
            out["confidence"] = float(out.get("confidence", 0.5))
            cache[key] = out
            return out
        except Exception as e:
            if attempt == retries - 1:
                print(f"  [failed] {e}")
                return {"label": NEG, "confidence": 0.0, "reason": f"error: {e}"}
            time.sleep(2 ** attempt)

results = []
t0 = time.time()
for i, row in diag.iterrows():
    results.append(classify(row["text"]))
    if (i + 1) % 25 == 0:
        json.dump(cache, open(CACHE, "w"))
        print(f"  {i+1}/{len(diag)}  ({time.time()-t0:.0f}s)")
json.dump(cache, open(CACHE, "w"))

diag["zs_label"] = [r["label"] for r in results]
diag["zs_conf"]  = [r["confidence"] for r in results]
diag["zs_reason"] = [r.get("reason", "") for r in results]
diag["zs_pred"]  = (diag["zs_label"] == POS).astype(int)
print(f"\nDone in {time.time()-t0:.0f}s")

In [ ]:
# ============================================================
# 3. Performance at the model's own decision
# ============================================================
y, p = diag["y"].values, diag["zs_pred"].values
print(classification_report(y, p, target_names=[NEG, POS], digits=3, zero_division=0))

cm = confusion_matrix(y, p)
print(pd.DataFrame(cm, index=[f"true: {NEG}", f"true: {POS}"],
                       columns=[f"pred: {NEG}", f"pred: {POS}"]).to_string())

print(f"\nrecall    {recall_score(y, p, zero_division=0):.3f}")
print(f"precision {precision_score(y, p, zero_division=0):.3f}")
print(f"discards  {100*(1-p.mean()):.1f}% of the corpus")
print(f"misses    {int(((y==1)&(p==0)).sum())} eligible articles")
print(f"base rate {y.mean():.3f}  ->  lift {precision_score(y,p,zero_division=0)/y.mean():.2f}x")

In [ ]:
# ============================================================
# 4. Threshold sweep on the returned confidence + WSS
# ============================================================
def wss_at_recall(y, conf, target=0.95):
    best = None
    for thr in np.linspace(0.001, 0.999, 999):
        pred = (conf >= thr).astype(int)
        rec = recall_score(y, pred, zero_division=0)
        if rec >= target:
            wss = (1 - pred.mean()) - (1 - rec)
            if best is None or wss > best[0]:
                best = (wss, thr, rec, pred.mean())
    return best

conf = diag["zs_conf"].values
rows = []
for thr in np.linspace(0.05, 0.95, 19):
    pr = (conf >= thr).astype(int)
    rows.append({"threshold": round(thr, 2),
                 "recall": round(recall_score(y, pr, zero_division=0), 3),
                 "precision": round(precision_score(y, pr, zero_division=0), 3),
                 "F1": round(f1_score(y, pr, zero_division=0), 3),
                 "pct_discarded": round(100*(1-pr.mean()), 1),
                 "eligible_missed": int(((y==1)&(pr==0)).sum())})
sweep = pd.DataFrame(rows)
sweep["lift"] = (sweep["precision"] / y.mean()).round(2)
print(sweep.to_string(index=False))

print()
for t in (0.95, 0.90, 0.85):
    r = wss_at_recall(y, conf, t)
    if r is None:
        print(f"WSS@{int(t*100)}: recall {t} UNREACHABLE")
    else:
        w, thr, rec, frac = r
        print(f"WSS@{int(t*100)}: {w*100:5.1f}%  (threshold {thr:.3f}, recall {rec:.3f})")
print("\nBenchmark (O'Mara-Eves et al. 2015): 30-70% saving at ~95% recall.")

In [ ]:
# ============================================================
# 5. Head-to-head with the fine-tuned model
#    Fill in the numbers printed by cell 14b/14c of the training notebook.
# ============================================================
FINETUNED = {"recall": 0.980, "precision": 0.449, "pct_discarded": 14.4, "wss95": 12.4}

zs = wss_at_recall(y, conf, 0.95)
zs_row = sweep.iloc[(sweep["recall"] - 0.95).abs().argmin()]
comparison = pd.DataFrame([
    {"approach": "Fine-tuned DeBERTa (synthetic)", **FINETUNED},
    {"approach": "Zero-shot LLM (no training)",
     "recall": zs_row["recall"], "precision": zs_row["precision"],
     "pct_discarded": zs_row["pct_discarded"],
     "wss95": round(zs[0]*100, 1) if zs else float("nan")},
])
print(comparison.to_string(index=False))

print("\nIf zero-shot wins, the screening problem is solved with no further annotation,")
print("and the prompt goes in the protocol as the eligibility criterion.")
print("If it loses, the difficulty is in the construct itself, not the model - and")
print("annotating labeling_seed.json becomes the necessary next step.")

In [ ]:
# ============================================================
# 6. Disagreement analysis - where the criterion is ambiguous
#    These are the abstracts to adjudicate by hand. They tell you whether
#    the prompt is wrong or the gold label is wrong.
# ============================================================
wrong = diag[diag["y"] != diag["zs_pred"]].copy()
print(f"Disagreements: {len(wrong)} of {len(diag)}\n")

fn = wrong[wrong["y"] == 1]
fp = wrong[wrong["y"] == 0]
print(f"MISSED eligible ({len(fn)}) - the expensive errors:")
for _, r in fn.head(6).iterrows():
    print(f"  conf {r['zs_conf']:.2f} | {r['zs_reason']}")
    print(f"     {r['text'][:150]}...")
print(f"\nFALSE alarms ({len(fp)}) - cheap, you just read them:")
for _, r in fp.head(4).iterrows():
    print(f"  conf {r['zs_conf']:.2f} | {r['zs_reason']}")
    print(f"     {r['text'][:150]}...")

diag.to_json(f"{OUT_DIR}/zeroshot_diagnostic_results.json",
             orient="records", indent=2, force_ascii=False)
print(f"\nSaved to {OUT_DIR}/zeroshot_diagnostic_results.json")

## 7. Running the full corpus

Only do this once the diagnostic numbers are acceptable. Point `corpus` at the parsed
Scopus export, reuse `classify`, and keep the cache — the whole corpus is a few dollars
and a couple of hours of wall time at one request per abstract.

Record in the protocol: the model identifier, the date, the exact prompt text, the
decision threshold, and the diagnostic performance. That combination is what makes the
screening reproducible and auditable, which a trained classifier's weights are not.